In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

In [5]:
import json
import time
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
MODEL = "gpt-4o-mini"

In [6]:
# messages = []
# messages.append()

In [7]:
messages = [
    SystemMessage(content = '당신은 친절한 AI 어시스턴트입니다.'),
    HumanMessage(content = "내 이름은 abc입니다. 반가워요")
]

response1 = llm.invoke(messages)
print(response1.content)

반가워요, abc님! 어떻게 도와드릴까요?


In [11]:
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content = '내 이름은 뭔가요?'))
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 이름은 abc입니다. 반가워요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반가워요, abc님! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름은 뭔가요?', additional_kwargs={}, response_metadata={})]

In [12]:
response2 = llm.invoke(messages)

In [13]:
print(response2.content)

abc님, 당신의 이름은 "abc"입니다! 다른 질문이 있으신가요?


In [16]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

def count_tokens(messages):
  total = 0
  for msg in messages:
    total += len(enc.encode(msg.content))
    total += 4
  return total

conversation = [SystemMessage(content = '당신은 친절한 AI 어시스턴트입니다.')]

sample_exchanges = [
    ('내 이름은 abc 입니다.', '반가워요, abc님! 어떻게 도와드릴까요?'),
    ('내 이름은 뭔가요?', 'abc님, 당신의 이름은 "abc"입니다! 다른 질문이 있으신가요?')

]

for i, (user_msg, ai_msg) in enumerate(sample_exchanges):
  conversation.append(HumanMessage(user_msg))
  conversation.append(AIMessage(ai_msg))
  tokens = count_tokens(conversation)

  print(f"{i} | {tokens} |")

0 | 47 |
1 | 83 |


In [ ]:
# 대화 window를 사용하는 법 -> 최신 맥락만 유지
# summarize : 10, 100 turn 마다 요약 -> 새로 대화를 시작하며 디테일 유지

In [19]:
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory

In [22]:
memory = ConversationBufferMemory(return_messages=True)

In [ ]:
# 내부적으로 memory를 가지고 있다가 사용 / 또는 summarize..: LangChain에서도 LangGraph처럼 state를 가지게 하려고 하고있고
  # 이러한 memory를 없애는 방향성으로 가는 중이라고 함.

# LangGraph : 모든 knowledge / 체인의 파이프 등을 하나의 state로 간주를 하면서 그래프를 연결해줌

In [23]:
memory.save_context(
    {'input': '안녕하세요. 저는 abc입니다.'},
    {'output': '안녕하세요 abc님! 만나서 반갑습니다.'}
)

memory.save_context(
      {'input': '오늘 날씨가 좋네요!'},
    {'output': '네, 정말 화창한 날씨입니다.'}

)

In [24]:
# memory라는 기억용 객체를 만들었고 맥락을 저장함(save_context)

In [26]:
history = memory.load_memory_variables({})

In [27]:
history

{'history': [HumanMessage(content='안녕하세요. 저는 abc입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요 abc님! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='네, 정말 화창한 날씨입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
# append를 썼던것보다, ConversationBufferMemory 객체를 통해 직관적으로 사용

In [28]:
for msg in history['history']:
  print(msg.content)

안녕하세요. 저는 abc입니다.
안녕하세요 abc님! 만나서 반갑습니다.
오늘 날씨가 좋네요!
네, 정말 화창한 날씨입니다.


In [30]:
# 만약 옵션이 False일 경우?
memory = ConversationBufferMemory(return_messages=False)
memory.save_context(
    {'input': '안녕하세요. 저는 abc입니다.'},
    {'output': '안녕하세요 abc님! 만나서 반갑습니다.'}
)

memory.save_context(
      {'input': '오늘 날씨가 좋네요!'},
    {'output': '네, 정말 화창한 날씨입니다.'}

)
history = memory.load_memory_variables({})
history

# return_messages=False는 포맷의 차이 (False면 텍스트만 필요할때)

{'history': 'Human: 안녕하세요. 저는 abc입니다.\nAI: 안녕하세요 abc님! 만나서 반갑습니다.\nHuman: 오늘 날씨가 좋네요!\nAI: 네, 정말 화창한 날씨입니다.'}

- join() 함수는 리스트나 튜플 같은 반복 가능한(iterable) 객체의 요소들을 하나의 문자열로 합칠 때 사용
- '구분자'.join(리스트) 형태로 사용

#### InMemoryChatMessageHistory

In [31]:
from langchain_core.chat_history import InMemoryChatMessageHistory

In [32]:
chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message('안녕하세요. 저는 abc입니다.')
chat_history.add_ai_message('안녕하세요 abc님! 만나서 반갑습니다.')
chat_history.add_user_message( '오늘 날씨가 좋네요!')
chat_history.add_ai_message('네, 정말 화창한 날씨입니다.')

In [33]:
chat_history

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕하세요. 저는 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 abc님! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}), AIMessage(content='네, 정말 화창한 날씨입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [34]:
for msg in chat_history.messages:
  print(msg.type, msg.content)

human 안녕하세요. 저는 abc입니다.
ai 안녕하세요 abc님! 만나서 반갑습니다.
human 오늘 날씨가 좋네요!
ai 네, 정말 화창한 날씨입니다.


In [36]:
chat_history.messages.append(HumanMessage(content = '내 이름이 뭔가요?'))
chat_history.messages

[HumanMessage(content='안녕하세요. 저는 abc입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요 abc님! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네, 정말 화창한 날씨입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭔가요?', additional_kwargs={}, response_metadata={})]

#### *는 파이썬의 언패킹(Unpacking) 연산자
- chat_history.messages는 그 자체로 [메시지1, 메시지2, ...]와 같은 리스트입니다. 만약 * 없이 [chat_history.messages, HumanMessage(...)]라고 쓰면 [[메시지1, 메시지2], HumanMessage]처럼 리스트 안에 리스트가 들어있는 구조가 됩니다.

- 하지만 *를 붙여주면 리스트라는 봉투를 까서 그 안의 알맹이(메시지 객체들)만 쏙쏙 꺼내어 새로운 리스트에 담아줍니다. 덕분에 LangChain이 원하는 평탄한(flat) 메시지 리스트 형태가 되어 정상적으로 동작하는 것입니다.

In [43]:
old_messages = ['msg1', 'msg2']
new_message = 'msg3'

# * 없이 넣었을 때 (중첩 리스트)
nested_list = [old_messages, new_message]
print("* 없을 때:", nested_list)

# * 를 사용했을 때 (언패킹)
flat_list = [*old_messages, new_message]
print("* 있을 때:", flat_list)

* 없을 때: [['msg1', 'msg2'], 'msg3']
* 있을 때: ['msg1', 'msg2', 'msg3']


In [39]:
messages = [
    *chat_history.messages, # 그냥 쓰면 안되고 *를 붙이면 element별로 풀어서 들어가므로 정상 작동됨
    HumanMessage(content = '내 이름이 뭔가요?')
]

llm.invoke(messages)

AIMessage(content='당신의 이름은 "abc"입니다. 다른 이름이 있으신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 78, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DNznskOBTDYob1uAVfZBlmF6G1pDg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f16-5b94-7690-88b4-223664574980-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 18, 'total_tokens': 96, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [46]:
# InMemoryChatMessageHistory를 사용해서
# 전자제품 매장의 고객 상담원과 고객의
# 채팅 히스토리와 마지막 쿼리를 짜보세요
'노트북을 사고 싶은데, 어떤 제품이 있나요?'
'가장 인기 있는 제품의 가격이 어떻게 되나요?'

chat_history = InMemoryChatMessageHistory()

chat_history.add_user_message('노트북을 사고 싶은데, 어떤 제품이 있나요?')
chat_history.add_ai_message('안녕하세요 고객님! 만나서 반갑습니다.')
chat_history.add_user_message('가장 인기 있는 제품의 가격이 어떻게 되나요?')
chat_history.add_ai_message('최근에 인기있는 제품군은 1백만원 중반대 입니다.')


messages = [
    *chat_history.messages,
    HumanMessage(content = '그럼 어떤 브랜드의 제품이 가장 인기있나요?')
]


In [52]:
llm.invoke(messages).content

'현재 인기 있는 노트북 브랜드로는 다음과 같은 것들이 있습니다:\n\n1. **애플 (Apple)** - MacBook Air, MacBook Pro\n2. **델 (Dell)** - XPS 시리즈, Inspiron 시리즈\n3. **레노버 (Lenovo)** - ThinkPad 시리즈, IdeaPad 시리즈\n4. **HP (휴렛팩커드)** - Spectre, Envy, Pavilion 시리즈\n5. **아수스 (ASUS)** - ZenBook, ROG (게임용 노트북)\n6. **에이서 (Acer)** - Swift 시리즈, Aspire 시리즈\n\n각 브랜드마다 다양한 모델과 사양이 있으니, 사용 용도에 맞춰 선택하는 것이 좋습니다. 별도의 요구 사항이나 예산이 있다면 더 구체적인 추천을 드릴 수 있습니다!'

In [50]:
for msg in chat_history.messages:
  print(msg.type, msg.content)

human 노트북을 사고 싶은데, 어떤 제품이 있나요?
ai 안녕하세요 고객님! 만나서 반갑습니다.
human 가장 인기 있는 제품의 가격이 어떻게 되나요?
ai 최근에 인기있는 제품군은 1백만원 중반대 입니다.


In [ ]:
# Runnable과 함께 사용시 조금 더 편하게 사용가능

#RunnablePassthrough, RunnableLambda, RunnableParallel
#RunnableWithMessageHistory

In [ ]:
# 기존에는 chain = prompt | llm | parser ..등등으로 실행했었음

In [55]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


In [57]:
# llm = ChatOpenAi..
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 AI 어시스턴트입니다.'),
    MessagesPlaceholder(variable_name='history'), # 히스토리가 들어갈 것(유저의 쿼리와 시스템 메시지 사이에)
    ('human', '{input}')
])

chain = prompt | llm

In [58]:
store = {}
# store = {'session_id1': ...., 'session_id2': ...., }

def get_session_history(session_id): # -> 어떤 함수? 세션 아이디를 받은 후,
  if session_id not in store: # session_id가 store에 없다면? (store는 뭔가 id가 누적되는 딕셔너리일 것)
    store[session_id] = InMemoryChatMessageHistory() # 없다면 history 객체 초기화해줌
  return store[session_id] # store에 저장된 값(session_id에 해당하는..)

In [59]:
chain_with_history = RunnableWithMessageHistory( # 필수적인 인자 4가지임
    chain, # 대화 기록을 연결할 체인(Chain)
    get_session_history, # 특정 세션 ID를 입력받았을 때,
                        # 세션의 과거 대화 기록(Chat History) 객체를 불러오거나 생성하는 함수
    input_messages_key = 'input', # 체인(Chain)을 실행(invoke)할 때
                                  # 사용자가 입력한 새로운 질문의 키값
    history_messages_key = 'history' # 과거 대화 기록들(Chat History)을 프롬프트 템플릿의 어느 변수에 주입할 것인지

)

In [62]:
config = {'configurable': {'session_id': 'user_001'}}
r1 = chain_with_history.invoke({'input': "안녕하세요, 제 이름은 AaBbCc입니다."}, config = config)

In [65]:
r1

AIMessage(content='안녕하세요, AaBbCc님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 36, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0EqL00FXeNvPyUtcwLwzRQUr1Xq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2f-dfe3-7b81-9f8f-77e3ad422ab8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 22, 'total_tokens': 58, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [66]:
r2 = chain_with_history.invoke({'input': "내 이름이 뭐라고 했죠?"}, config = config)

In [67]:
r2

AIMessage(content='당신의 이름은 AaBbCc입니다. 맞나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 74, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0Fd8NnF0f2fjLveUZ6V1SNxdJuE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f30-9e39-7143-86c2-49a6c347dc73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 13, 'total_tokens': 87, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [69]:
store # 'user_001' 등, 대화 내역이 쌓여있는 것을 볼 수 있음

{'user_001': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕하세요, 제 이름은 AaBbCc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, AaBbCc님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 36, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0EqL00FXeNvPyUtcwLwzRQUr1Xq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2f-dfe3-7b81-9f8f-77e3ad422ab8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 22, 'total_tokens': 58, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

In [82]:
r3 = chain_with_history.invoke({'input': "오늘 뭐하면 좋을까요?"}, config = config)

In [83]:
r3.content

'오늘 무엇을 할지 고민 중이시군요! 몇 가지 제안을 드릴게요:\n\n1. **산책하기**: 가까운 공원이나 자연에서 산책하며 상쾌한 공기를 마시는 것은 항상 좋은 선택입니다.\n2. **책 읽기**: 읽고 싶었던 책이나 새로운 장르의 책에 도전해보세요.\n3. **영화/드라마 감상**: 보고 싶었던 영화나 드라마를 즐기는 시간을 가질 수 있습니다.\n4. **요리하기**: 새로운 요리 레시피에 도전해보는 것도 재미있습니다.\n5. **취미 활동**: 그림 그리기, 악기 치기, DIY 프로젝트 등 평소에 해보고 싶었던 취미에 도전해보세요.\n\n어떤 활동이 가장 끌리시나요?'

In [84]:
# RunnableWithHisttory를 이용해 첫 실습을 바꿀것
"""
chat_history = InMemoryChatMessageHistory()

chat_history.add_user_message('노트북을 사고 싶은데, 어떤 제품이 있나요?')
chat_history.add_ai_message('안녕하세요 고객님! 만나서 반갑습니다.')
chat_history.add_user_message('가장 인기 있는 제품의 가격이 어떻게 되나요?')
chat_history.add_ai_message('최근에 인기있는 제품군은 1백만원 중반대 입니다.')


messages = [
    *chat_history.messages,
    HumanMessage(content = '그럼 어떤 브랜드의 제품이 가장 인기있나요?')
]

"""
store = {}
def get_session_history(session_id): # -> 어떤 함수? 세션 아이디를 받은 후,
  if session_id not in store: # session_id가 store에 없다면? (store는 뭔가 id가 누적되는 딕셔너리일 것)
    store[session_id] = InMemoryChatMessageHistory() # 없다면 history 객체 초기화해줌
  return store[session_id] # store에 저장된 값(session_id에 해당하는..)

# ===============

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 노트북 판매원입니다..'),
    MessagesPlaceholder(variable_name='history'), # 히스토리가 들어갈 것(유저의 쿼리와 시스템 메시지 사이에)
    ('human', '{input}')
])

chain = prompt | llm

# ===============

chain_with_history = RunnableWithMessageHistory( # 필수적인 인자 4가지임
    chain, # 대화 기록을 연결할 체인(Chain)
    get_session_history, # 특정 세션 ID를 입력받았을 때,
                        # 세션의 과거 대화 기록(Chat History) 객체를 불러오거나 생성하는 함수
    input_messages_key = 'input', # 체인(Chain)을 실행(invoke)할 때
                                  # 사용자가 입력한 새로운 질문의 키값
    history_messages_key = 'history' # 과거 대화 기록들(Chat History)을 프롬프트 템플릿의 어느 변수에 주입할 것인지

)

# ===============

config = {'configurable': {'session_id': 'user_260327'}}
customer1 = chain_with_history.invoke({'input': "노트북을 사고 싶은데, 어떤 제품이 있나요?"}, config = config)

In [86]:
customer1.content

'노트북을 구매하시려면 고려해야 할 여러 옵션이 있습니다. 사용 용도에 따라 추천해드릴 수 있는 노트북이 달라질 수 있습니다. 주로 사용하실 용도가 무엇인지 알려주시면 더 구체적으로 추천해드릴 수 있습니다. 일반적으로 고려할 수 있는 노트북의 종류는 다음과 같습니다:\n\n1. **일반적인 사용 (웹서핑, 문서작업 등)**:\n   - **HP Pavilion 시리즈**\n   - **Dell Inspiron 시리즈**\n   - **Lenovo IdeaPad 시리즈**\n\n2. **학생 및 학습용**:\n   - **MacBook Air**: 디자인과 휴대성이 뛰어나고, 교육용으로 많이 사용됩니다.\n   - **Chromebook**: 웹 기반 작업에 최적화된 저렴한 옵션입니다.\n\n3. **게이밍용**:\n   - **Asus ROG 시리즈**: 고성능 그래픽 카드와 빠른 프로세서를 제공합니다.\n   - **MSI GAMING 시리즈**: 뛰어난 성능과 쿨링 시스템이 장점입니다.\n\n4. **전문가 및 크리에이터용**:\n   - **MacBook Pro**: 디자인, 영상 편집 등 고사양 작업에 적합합니다.\n   - **Dell XPS 시리즈**: 높은 해상도와 강력한 성능을 갖춘 제품입니다.\n\n5. **2-in-1 컨버터블 노트북**:\n   - **Microsoft Surface Pro**: 태블릿처럼 사용할 수 있는 노트북입니다.\n   - **Lenovo Yoga 시리즈**: 다양한 모드로 변형 가능한 제품입니다.\n\n어떤 용도로 사용할지에 따라 예산과 선호하는 브랜드도 알려주시면 더욱 맞춤형 조언을 드릴 수 있습니다!'

In [87]:
customer2 = chain_with_history.invoke({'input': "가장 인기 있는 제품의 가격이 어떻게 되나요?"}, config = config)

In [88]:
customer2.content

'각 브랜드와 모델마다 가격대가 다양하지만, 다음은 대중적으로 인기 있는 몇 가지 노트북 모델과 대략적인 가격대입니다 (2023년 기준):\n\n1. **HP Pavilion x360**:\n   - 가격대: 약 60만원 ~ 100만원\n\n2. **Dell XPS 13**:\n   - 가격대: 약 100만원 ~ 200만원\n\n3. **Apple MacBook Air (M2)**:\n   - 가격대: 약 130만원 ~ 170만원\n\n4. **Asus ROG Zephyrus G14 (게이밍 노트북)**:\n   - 가격대: 약 150만원 ~ 250만원\n\n5. **Lenovo IdeaPad 3**:\n   - 가격대: 약 50만원 ~ 80만원\n\n6. **Microsoft Surface Laptop 4**:\n   - 가격대: 약 100만원 ~ 150만원\n\n위의 가격들은 대략적인 평균이며, 지역이나 판매처에 따라 다를 수 있습니다. 원하는 기능이나 사양에 따라 차이가 나기 때문에, 구체적인 모델을 선택할 때는 여러 판매처의 가격과 사양을 비교하는 것이 좋습니다. 또한, 할인 행사나 프로모션이 있을 때 구매하면 더 저렴한 가격에 제품을 구입할 수 있는 기회가 많습니다. 더 궁금한 점이나 특정 모델에 대한 정보가 필요하시면 말씀해 주세요!'

In [90]:
len(store['user_260327'].messages)

4

In [91]:
for msg in store['user_260327'].messages:
  prefix = 'customer' if msg.type == 'human' else 'sales person'
  print(f"{prefix} - {msg.content[:30]}")

customer - 노트북을 사고 싶은데, 어떤 제품이 있나요?
sales person - 노트북을 구매하시려면 고려해야 할 여러 옵션이 있습니다
customer - 가장 인기 있는 제품의 가격이 어떻게 되나요?
sales person - 각 브랜드와 모델마다 가격대가 다양하지만, 다음은 대중


#### ConversationBufferWindowMemory

In [93]:
window_memory = ConversationBufferWindowMemory(k=2, return_messages = True) # k는 가져오는 갯수 넣는 파라미터 (최신 대화중에 k개만 유지한다)

In [94]:
window_memory.save_context({'input': '첫번째, 내 이름은 abc입니다.'}, {'output':'안녕하세요 abc님'})
window_memory.save_context({'input': '두번째, 나는 학생'}, {'output':'학생이군요'})
window_memory.save_context({'input': '세번째, 나는 파이썬 좋아함.'}, {'output':'파이썬 좋아하는군요'})

In [95]:
window_memory.load_memory_variables({})

{'history': [HumanMessage(content='두번째, 나는 학생', additional_kwargs={}, response_metadata={}),
  AIMessage(content='학생이군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='세번째, 나는 파이썬 좋아함.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='파이썬 좋아하는군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [99]:
from langchain_core.messages import trim_messages

messages = []

conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요, abc님'),
    ('두번째, 나는 학생', '학생이군요'),
    ('세번째, 나는 파이썬 좋아함.', '파이썬 좋아하는군요')
]

for user_msg, ai_msg in conversations:
  messages.append(HumanMessage(content=user_msg))
  messages.append(AIMessage(content=ai_msg))

messages

[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='두번째, 나는 학생', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬 좋아함.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬 좋아하는군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [100]:
trimmer = trim_messages(max_tokens= 4, strategy='last', token_counter=len,start_on='human')
trimmed = trimmer.invoke(messages)

In [103]:
trimmed

[HumanMessage(content='두번째, 나는 학생', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬 좋아함.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬 좋아하는군요', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [104]:
messages = []
K = 2
def add_turn(user_input, ai_output):
  global messages
  messages.append(HumanMessage(content=user_input))
  messages.append(AIMessage(content=ai_output))

  messages = messages[-(K*2):]

In [108]:
conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요, abc님'),
    ('두번째, 나는 학생', '학생이군요'),
    ('세번째, 나는 파이썬 좋아함.', '파이썬 좋아하는군요')
]

add_turn('첫번째, 내 이름은 abc입니다.' ,'안녕하세요, abc님')

print('11111111')
print(messages)
add_turn('두번쨰, 나는 학생입니다.','학생이군요 멋지십니다.')
print('222222')
print(messages)

11111111
[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
222222
[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='두번쨰, 나는 학생입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='학생이군요 멋지십니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [107]:
from langchain_classic.memory import ConversationSummaryMemory

In [111]:
summary_llm = ChatOpenAI(model = 'gpt-4o-mini')
# 대화를 요약하는 작업 자체도 지능이 필요하기 때문에 LLM이 필요
summary_memory = ConversationSummaryMemory(
  # 대화를 있는 그대로 저장하지 않고 요약해서 저장하는 ConversationSummaryMemory
    llm = summary_llm,
    return_messages = False
)

conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요, abc님'),
    ('두번째, 나는 학생', '학생이군요'),
    ('세번째, 나는 파이썬 좋아함.', '파이썬 좋아하는군요')
]

# 대화 3세트를 save_context로 하나씩 메모리에 넣음
# 대화가 하나 들어올 때마다 summary_llm이 작동하여
# "기존 요약본 + 방금 들어온 새 대화"를 읽고 새로운 요약본으로 덮어쓰기(업데이트)를 합니다.
for user_msg, ai_msg in conversations:
  summary_memory.save_context({'input': user_msg}, {'output': ai_msg})

In [113]:
result = summary_memory.load_memory_variables({})

In [114]:
result['history']

'The human introduces themselves as abc, and the AI greets them by name. The human states they are a student, and the AI acknowledges this. The human also mentions their fondness for Python, which the AI recognizes.'

#### SummaryMemory 클래스 구현

- 일정 턴(기본 3턴)마다 누적된 대화를 LLM을 이용해 하나의 간결한 요약문으로 자동 압축(업데이트)합니다.
- 생성된 '과거 대화 요약본'과 아직 요약되지 않은 '최근 대화'를 결합하여 제공함으로써, 메모리(토큰)를 절약하면서도 긴 대화의 문맥을 효과적으로 유지하게 해줍니다.

- (참고: 즉, LangChain의 ConversationSummaryMemory가 내부적으로 어떻게 작동하는지 직접 원리를 구현해 본 커스텀 클래스입니다!)

In [118]:
class SummaryMemory:

  def __init__(self, summary_interval = 3):
    self.summary = ''
    self.recent_messages = []
    self.summary_interval = summary_interval
    self.turn_count = 0

  def _summarize(self, text):
    response = llm.invoke([
        SystemMessage(content = '주어진 대화 내용을 간결하게 한국어로 요약하세요.'),
        HumanMessage(content = f'기존 요약:\n{self.summary}\n\n새 대화:\n{text}')
    ])
    return response.content

  def add_exchange(self, user_msg, ai_msg):
    self.recent_messages.append(f'사용자: {user_msg}')
    self.recent_messages.append(f'AI: {user_msg}')
    self.turn_count += 1
                        # %는 나머지를 구하는거
    if self.turn_count % self.summary_interval ==0: # 계속 append 하다가 interval마다 요약
      conversation_text = '\n'.join(self.recent_messages)
      self.summary = self._summarize(conversation_text) # 요약
      self.recent_messags = []
      print(f'요약 완료 턴 {self.turn_count}에서 요약')

  def get_context(self):
    parts = []
    if self.summary:
      parts.append(f'[이전 대화 요약] {self.summary}')
    if self.recent_messages:
      parts.append(f'[최근 대화]\n + \n'.join(self.recent_messages))
    return "\n\n".join(parts)


In [119]:
smem = SummaryMemory(summary_interval=2)

In [122]:

conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요, abc님'),
    ('두번째, 나는 학생', '학생이군요'),
    ('세번째, 나는 파이썬 좋아함.', '파이썬 좋아하는군요')
]

for user_msg, ai_msg in conversations:
  smem.add_exchange(user_msg, ai_msg)

요약 완료 턴 2에서 요약


In [123]:
smem.get_context()

'[이전 대화 요약] 대화 요약: 사용자가 "내 이름은 abc입니다"와 "나는 학생"이라고 말하자, AI가 동일하게 반복함.\n\n사용자: 첫번째, 내 이름은 abc입니다.[최근 대화]\n + \nAI: 첫번째, 내 이름은 abc입니다.[최근 대화]\n + \n사용자: 두번째, 나는 학생[최근 대화]\n + \nAI: 두번째, 나는 학생[최근 대화]\n + \n사용자: 세번째, 나는 파이썬 좋아함.[최근 대화]\n + \nAI: 세번째, 나는 파이썬 좋아함.'

In [138]:
class SummaryChatbot:
  def __init__(self, system_prompt='당신은 도움이 되는 AI 어시스턴트입니다.', summary_interval=3):
    self.system_prompt = system_prompt
    self.memory = SummaryMemory(summary_interval=summary_interval)

  def chat(self, user_input):
    context = self.memory.get_context()

    messages = [
        SystemMessage(content = self.system_prompt)
    ]
    if context:
      messages.append(SystemMessage(content = f'대화 맥락: \n{context}'))

    messages.append(HumanMessage(content=user_input))

    response = llm.invoke(messages)
    ai_response = response.content
    self.memory.add_exchange(user_input, ai_response)

    return ai_response

In [139]:
bot = SummaryChatbot(
    system_prompt = '당신은 IT 커리어 상담사입니다.',
    summary_interval = 2
)

In [142]:
questions =[
    '안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다.',
    'AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?',

]

In [143]:
for q in questions:
  print(f'[user] {q}')
  answer = bot.chat(q)
  print(f'[상담사] {answer}')

[user] 안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다.
[상담사] 안녕하세요! 백엔드 개발자로 3년의 경력을 쌓으셨군요. 어떤 고민이 있으신가요? AI/ML 분야로의 전환에 대해 더 이야기해 보실까요?
[user] AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?
요약 완료 턴 4에서 요약
[상담사] AI/ML 분야로의 전환을 준비하기 위해서는 여러 가지 단계를 고려할 수 있습니다. 다음은 그 과정에서 유용할 수 있는 몇 가지 단계입니다:

1. **기초 수학 및 통계학**:
   - 선형 대수학, 미적분학, 확률 및 통계에 대한 이해가 중요합니다. 이러한 기초는 AI와 머신러닝 알고리즘의 이론적 배경을 이해하는 데 도움이 됩니다.

2. **프로그래밍 스킬**:
   - 이미 백엔드 개발자로서 프로그래밍 경험이 있으므로, Python에 대한 익숙함을 갖추는 것이 좋습니다. Python은 AI/ML 분야에서 널리 사용되는 언어입니다. 데이터 처리 및 분석을 위한 라이브러리(Pandas, NumPy 등)와 머신러닝 라이브러리(Scikit-learn, TensorFlow, PyTorch 등)를 익히세요.

3. **기계 학습 이론**:
   - 머신러닝의 다양한 알고리즘과 개념(회귀, 분류, 군집화 등)을 공부하세요. Coursera, edX, Udacity 등의 온라인 플랫폼에서 AI/ML 관련 강의를 수강할 수 있습니다.

4. **프로젝트 경험**:
   - 실제 데이터셋을 사용하여 작은 프로젝트를 진행해보는 것이 좋습니다. Kaggle과 같은 플랫폼에서 다양한 데이터셋을 활용하여 실전 경험을 쌓을 수 있습니다.

5. **딥러닝**:
   - 기계 학습을 이해한 후, 딥러닝에 대한 이해를 넓히는 것이 좋습니다. TensorFlow 또는 PyTorch를 사용하여 신경망을 구축해보세요.

6. **전문 커뮤니티 참여**:
   - 관련 분야의 컨퍼런스, 세미나 또는 온라인 포럼에 참여하여 최신 동향을 파악하고 네트워킹을 할 수 있습니다